# J2 bis · Test de CLIP sur le jeu Wikimedia

But : mesurer si la validation « est-ce bien un cours d'eau ? » du J2 fonctionne sur des photos **jamais vues**.

- Même modèle, mêmes phrases et même seuil (0,35) que le J2 : **on ne règle rien ici**.
- Les photos marquées `utilisable = oui` sont des cours d'eau, sans ambiguïté.
- Les photos marquées `non` mélangent des hors-sujet (manuscrits, cartes, bâtiments…) et de vraies rivières écartées pour d'autres raisons (vue aérienne, lieu déjà photographié deux fois). Section 3 : on les trie à la main, **sans voir la réponse de CLIP**.

⚙️ Ce notebook tourne sur **CPU** (Exécution → Modifier le type d'exécution → CPU) : le GPU reste à l'API.

## 0. Drive et photos

In [ ]:
import os, csv, math, torch, numpy as np
from PIL import Image, ImageOps
from google.colab import drive
drive.mount('/content/drive')

DOSSIER = "/content/drive/MyDrive/streamsentinel/eval_j4"
FICHIER_VERITE = f"{DOSSIER}/verite_clip.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"

photos = sorted(f for f in os.listdir(f"{DOSSIER}/photos") if f.lower().endswith((".jpg", ".jpeg", ".png")))

utilisable = {}
for nom in ("miryame", "frere"):
    chemin = f"{DOSSIER}/annotations_{nom}.csv"
    if os.path.exists(chemin):
        for l in csv.DictReader(open(chemin, encoding="utf-8")):
            if l["utilisable"] == "oui" or l["fichier"] not in utilisable:
                utilisable[l["fichier"]] = l["utilisable"]
print(len(photos), "photos,", sum(v == "oui" for v in utilisable.values()), "marquées utilisables")

def charger(nom, taille=1024):
    img = ImageOps.exif_transpose(Image.open(f"{DOSSIER}/photos/{nom}")).convert("RGB")
    img.thumbnail((taille, taille))
    return img


## 1. CLIP, identique au J2

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

def _vecteur(sortie, projection):
    v = sortie if isinstance(sortie, torch.Tensor) else sortie.pooler_output
    if v.shape[-1] != projection.out_features:
        v = projection(v)
    return v / v.norm(dim=-1, keepdim=True)

@torch.no_grad()
def vecteur_image(img):
    e = clip_processor(images=img, return_tensors="pt").to(device)
    return _vecteur(clip_model.get_image_features(**e), clip_model.visual_projection)

@torch.no_grad()
def vecteurs_texte(phrases):
    e = clip_processor(text=phrases, return_tensors="pt", padding=True, truncation=True).to(device)
    return _vecteur(clip_model.get_text_features(**e), clip_model.text_projection)

SCENES = {
    "cours_deau": ["a photo of a river", "a photo of a small stream in nature",
                   "a photo of an urban canal", "a photo of a creek with rocks"],
    "plan_deau":  ["a photo of a lake", "a photo of the sea and a beach",
                   "a photo of a swimming pool"],
    "hors_sujet": ["a photo of a street with cars", "a photo of a parking lot",
                   "a photo of a building", "an indoor photo of a room",
                   "a portrait photo of a person", "a photo of a forest without water",
                   "a screenshot of a document or a map"],
}
PHRASES, GROUPES = [], []
for g, ph in SCENES.items():
    PHRASES += ph
    GROUPES += [g] * len(ph)
VECTEURS_SCENES = vecteurs_texte(PHRASES)
SEUIL = 0.35   # seuil du J2, volontairement inchangé

def valider_scene(img):
    probas = torch.softmax((vecteur_image(img) @ VECTEURS_SCENES.T)[0] * 100, dim=0).cpu().numpy()
    score = float(sum(p for g, p in zip(GROUPES, probas) if g == "cours_deau"))
    return score >= SEUIL, round(score, 3), PHRASES[int(np.argmax(probas))]

clip_resultats = {}
for nom in photos:
    try:
        clip_resultats[nom] = valider_scene(charger(nom))
    except Exception as e:
        print("Ignorée :", nom, type(e).__name__)
print(len(clip_resultats), "photos analysées par CLIP")


## 2. Vérité terrain

Les photos `utilisable = oui` sont automatiquement des cours d'eau. Pour les autres, clique sur la bonne réponse :

- **Cours d'eau** : on voit une rivière, un ruisseau ou un canal, même en vue aérienne ou déjà photographié ;
- **Hors sujet** : manuscrit, carte, bâtiment seul, personne, lac, mer, forêt sans eau… ;
- **Je ne sais pas** : la photo est exclue du calcul.

Chaque clic est enregistré dans `verite_clip.csv` : on peut s'arrêter et reprendre.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

verite = {n: "cours_deau" for n, u in utilisable.items() if u == "oui"}
if os.path.exists(FICHIER_VERITE):
    verite.update({l["fichier"]: l["verite"] for l in csv.DictReader(open(FICHIER_VERITE, encoding="utf-8"))})

a_trier = [n for n in photos if n in clip_resultats and n not in verite]
print(len(a_trier), "photos à trier")

def enregistrer():
    with open(FICHIER_VERITE, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["fichier", "verite"])
        for n, v in verite.items():
            w.writerow([n, v])

sortie = widgets.Output()
etat = {"i": 0}

def afficher():
    with sortie:
        clear_output(wait=True)
        if etat["i"] >= len(a_trier):
            print("Tri terminé. Lance la section 3.")
            return
        nom = a_trier[etat["i"]]
        print(f"{etat['i'] + 1}/{len(a_trier)} · {nom}")
        display(charger(nom, 600))

def choisir(valeur):
    def action(_):
        if etat["i"] < len(a_trier):
            verite[a_trier[etat["i"]]] = valeur
            enregistrer()
            etat["i"] += 1
            afficher()
    return action

boutons = [widgets.Button(description=t) for t in ("Cours d'eau", "Hors sujet", "Je ne sais pas")]
for b, v in zip(boutons, ("cours_deau", "hors_sujet", "?")):
    b.on_click(choisir(v))
display(widgets.HBox(boutons), sortie)
afficher()


## 3. Résultats

- **Sensibilité** : part des vrais cours d'eau que CLIP accepte. Un rejet à tort oblige le citoyen à reprendre sa photo.
- **Spécificité** : part des hors-sujet que CLIP rejette. Une acceptation à tort envoie une photo inutile dans le pipeline.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    d = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0, c - d), min(1, c + d)

vp = fn = vn = fp = 0
erreurs = []
for nom, v in verite.items():
    if v == "?" or nom not in clip_resultats:
        continue
    accepte, score, scene = clip_resultats[nom]
    if v == "cours_deau":
        vp += accepte; fn += not accepte
        if not accepte: erreurs.append(("cours d'eau rejeté", nom, score, scene))
    else:
        vn += not accepte; fp += accepte
        if accepte: erreurs.append(("hors-sujet accepté", nom, score, scene))

for titre, k, n in (("Sensibilité (cours d'eau acceptés)", vp, vp + fn),
                    ("Spécificité (hors-sujet rejetés)", vn, vn + fp),
                    ("Exactitude globale", vp + vn, vp + fn + vn + fp)):
    b, h = wilson(k, n)
    print(f"{titre:36s} {k}/{n} = {k / n:.0%}  [{b:.0%}–{h:.0%}]" if n else f"{titre}: aucune photo")

print("\nErreurs :")
for type_err, nom, score, scene in sorted(erreurs):
    print(f"  {type_err:20s} {nom[:45]:45s} score {score:.2f} · scène vue : {scene}")
